In [1]:
import torch

def naive_softmax(x: torch.Tensor) -> torch.Tensor:
    return x.exp() / x.exp().sum()

/home/recoverx/astarag/ring-flash-attention/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
x = torch.randn(10)  # generate normally distributed random numbers
a = torch.softmax(x, dim=-1) # reference output
b = naive_softmax(x) # our naive version

print("a", a)
print("b", b)
print("allclose", torch.allclose(a, b, atol=1e-6))

a tensor([0.0201, 0.1774, 0.0251, 0.3586, 0.1905, 0.0601, 0.0289, 0.0779, 0.0229,
        0.0385])
b tensor([0.0201, 0.1774, 0.0251, 0.3586, 0.1905, 0.0601, 0.0289, 0.0779, 0.0229,
        0.0385])
allclose True


In [3]:
naive_softmax(x * 100)

tensor([0., nan, 0., nan, nan, 0., 0., 0., 0., 0.])

In [4]:
x = torch.randn(10)

x1,x2 = torch.chunk(x, 2)
s1 = naive_softmax(x1)
s2 = naive_softmax(x2)

print("We have:")
print(f"s1 = {s1}")
print(f"s2 = {s2}")

target = naive_softmax(x)
print("We want:")
print(f"target = {target}")

We have:
s1 = tensor([0.1045, 0.2665, 0.2073, 0.1694, 0.2522])
s2 = tensor([0.2010, 0.1274, 0.3327, 0.1338, 0.2051])
We want:
target = tensor([0.0468, 0.1193, 0.0928, 0.0759, 0.1129, 0.1110, 0.0703, 0.1838, 0.0739,
        0.1133])


In [5]:
sum_exp_x1 = x1.exp().sum()
sum_exp_x2 = x2.exp().sum()

s1_corrected = s1 * sum_exp_x1 / (sum_exp_x1 + sum_exp_x2)
s2_corrected = s2 * sum_exp_x2 / (sum_exp_x1 + sum_exp_x2)

print("After correction with help of sum_exp values:")
s_combined = torch.cat([s1_corrected, s2_corrected])
print("s_combined", s_combined)

print("allclose(s_combined, target):", torch.allclose(s_combined, target))

After correction with help of sum_exp values:
s_combined tensor([0.0468, 0.1193, 0.0928, 0.0759, 0.1129, 0.1110, 0.0703, 0.1838, 0.0739,
        0.1133])
allclose(s_combined, target): True


In [6]:
from typing import Tuple, Sequence

def naive_softmax2(x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    sum_exp = x.exp().sum()
    return x.exp() / sum_exp, sum_exp

In [8]:
def naive_blockwise_softmax(blocks: Sequence[torch.Tensor]) -> torch.Tensor:
    total_sum_exp = 0
    blocks_out = []
    for block in blocks:
        block_softmax, block_sum_exp = naive_softmax2(block)
        blocks_out.append((block_softmax, block_sum_exp))
        total_sum_exp += block_sum_exp

    out = []
    for block_softmax, block_sum_exp in blocks_out:
        out.append(block_softmax * block_sum_exp / total_sum_exp)
    return torch.cat(out)

x_long = torch.randn(20)
chunks = torch.chunk(x_long, 4)
a = naive_blockwise_softmax(chunks)
b = torch.softmax(x_long, dim=-1)
print("a", a)
print("b", b)
print("allclose:", torch.allclose(a, b))

a tensor([0.0223, 0.0517, 0.0057, 0.0023, 0.0167, 0.0223, 0.0398, 0.0322, 0.0167,
        0.2648, 0.0876, 0.0192, 0.0071, 0.3368, 0.0193, 0.0128, 0.0322, 0.0055,
        0.0016, 0.0032])
b tensor([0.0223, 0.0517, 0.0057, 0.0023, 0.0167, 0.0223, 0.0398, 0.0322, 0.0167,
        0.2648, 0.0876, 0.0192, 0.0071, 0.3368, 0.0193, 0.0128, 0.0322, 0.0055,
        0.0016, 0.0032])
allclose: True


In [9]:
x = torch.randn(8)
print(naive_softmax(x))
print(naive_softmax(x+5))
print(naive_softmax(x-3))

tensor([0.1091, 0.0059, 0.0492, 0.2735, 0.0961, 0.1342, 0.0438, 0.2882])
tensor([0.1091, 0.0059, 0.0492, 0.2735, 0.0961, 0.1342, 0.0438, 0.2882])
tensor([0.1091, 0.0059, 0.0492, 0.2735, 0.0961, 0.1342, 0.0438, 0.2882])


In [10]:
def stable_softmax(x):
    m = x.max()
    return (x-m).exp() / (x-m).exp().sum()

In [11]:
def stable_softmax2(x):
    """returns softmax result and log sum exp"""
    m = x.max()
    a = (x - m).exp()
    b = a.sum()
    lse = m + torch.log(b)
    return a / b, lse

In [12]:
x = torch.randn(20)

a = torch.softmax(x, dim=-1)

x1, x2 = x.chunk(2)

b1, lse1 = stable_softmax2(x1)
b2, lse2 = stable_softmax2(x2)

c1 = b1 * torch.exp(lse1) / (torch.exp(lse1) + torch.exp(lse2))
c2 = b2 * torch.exp(lse2) / (torch.exp(lse1) + torch.exp(lse2))

print(a)
print(torch.cat([c1, c2]), torch.allclose(a, torch.cat([c1, c2])))

tensor([0.0306, 0.0157, 0.0914, 0.0853, 0.0171, 0.1437, 0.0312, 0.1190, 0.0202,
        0.0724, 0.0117, 0.0761, 0.0070, 0.0469, 0.0174, 0.0115, 0.1093, 0.0083,
        0.0435, 0.0416])
tensor([0.0306, 0.0157, 0.0914, 0.0853, 0.0171, 0.1437, 0.0312, 0.1190, 0.0202,
        0.0724, 0.0117, 0.0761, 0.0070, 0.0469, 0.0174, 0.0115, 0.1093, 0.0083,
        0.0435, 0.0416]) True


In [ ]:
d1 = 